In [ ]:
# --- Welcome to Joe's Kinetic Analyser 3000 --- #

#This tool kit is capable of extracting rate constants from raw data, and the following notebook can model the impacts of descriptor variables on rate constants#
# Follow the steps in the cell below, and read the Notes
# --- Imports --- #
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, KFold, cross_val_predict
from sklearn.metrics import r2_score

In [ ]:
# --- User Changed Settings Go Here --- #

path = r"xxx"      #input folder directory with raw kinetics data CSV files 
file_prefix = "prefix_"         #input the CSV file prefix name
time_column = "time_s"        #input time column name from CSVs
conc_column = "conc"        #input concentration column name from CSV
N_runs = 10                #input the number of runs
k_guess = 0.001            # initial k estimate
A0_guess = 0.002                # rough starting concentraion
measured_species = "A"       #input if reactant "A" or product "P" is measured
order = 1                   #input the order of reaction here, change until curve is fit well

# --- Notes --- #
#Time units should be in seconds in the CSV
#concentration should be in M
#k_guess and A0_guess can be sampled and checked against fitting results for making sure results are close to actual values.
#A large initial k_guess may yield a poor fit
#work to come on raising errors on poor fits for usability

In [ ]:
# --- Read Each CSV File --- #
dfs = {}

for i in range(1,N_runs +1):
    file_name = f"{file_prefix}{i}.csv"
    path_csv = os.path.join(path, file_name)
    dfs[f"df_{i}"] = pd.read_csv(path_csv)

full_df = dfs["df_1"][[time_column, conc_column]].copy()
full_df = full_df.rename(columns = {conc_column: "Run_1"})

for n in range(2, N_runs + 1):
    add_df = dfs[f"df_{n}"][[time_column, conc_column]].copy()
    add_df = add_df.rename(columns={conc_column: f"Run_{n}"})
    full_df = pd.merge(full_df, add_df, on=time_column, how="outer")

print(full_df)
    

In [ ]:
# --- Plot each Kinetic run --- #
for i in range (1, N_runs + 1):
    plt.figure()
    plt.scatter(full_df[time_column], full_df[f"Run_{i}"], color = 'blue')
    plt.title(f"Run_{i}")
    plt.xlabel("time /s")
    plt.ylabel("conc /M")
    plt.show()

In [ ]:
# --- NLs Fit --- # 

def zero_order_P(time, k, A0_fit):                                  
    return np.minimum(k * time, A0_fit)

def first_order_P(time, k, A0_fit):
    return A0_fit * (1 - np.exp(-k * time))

def second_order_P(time, k, A0_fit):
    return A0_fit - (A0_fit / (1 + A0_fit * k * time))

def zero_order_A(time, k, A0_fit):
    return np.maximum(A0_fit -k * time, 0)

def first_order_A(time, k, A0_fit):
    return A0_fit * np.exp(-k * time)

def second_order_A(time, k, A0_fit):
    return A0_fit / (1 + A0_fit * k * time)


def order_choice(order, measured_species):
    if measured_species == "P":
        if order == 0:
            return zero_order_P
        elif order == 1:
            return first_order_P
        elif order == 2:
            return second_order_P
    elif measured_species == "A":
        if order == 0:
            return zero_order_A
        elif order == 1:
            return first_order_A
        elif order == 2:
            return second_order_A
            
    raise ValueError("order must be 0,1 or 2, and/or measured_species must be A or P") 
            


f = order_choice(order, measured_species)
all_k_values = []

for i in range(1,N_runs+1):
    Working_df = full_df[[time_column,f"Run_{i}"]].dropna()
    x = Working_df[time_column]
    y = Working_df[f"Run_{i}"]

    #k_guess = init_k(x, y, order)
    nls = curve_fit(f, x, y, p0=[k_guess, A0_guess])
    k_est_array = nls[0]
    k_est = k_est_array[0]
    A0_est = nls[0][1]
    all_k_values.append({'Run': f"Run_{i}",
                         "k": k_est,
                         "A0": A0_est})

table = pd.DataFrame(all_k_values)
table

# --- This writes your k data into a CSV for future use --- #
#table.to.csv(os.path.join(path "k_values.csv", index = False)
    

In [ ]:
# --- The Plotter 3000 --- #


for i in range(1,N_runs + 1):
    X = full_df[time_column]
    y = full_df[f"Run_{i}"]
    k_fit = table.loc[i-1, "k"]
    A0_fit = table.loc[i-1, "A0"]
    x_smooth =np.linspace(0, full_df[time_column].max(), num = 1000)
    

    plt.figure()
    plt.scatter(X, y, color='blue')
    plt.plot(x_smooth, f(x_smooth, k_fit, A0_fit))
    plt.title(f"Run_{i}, k = {k_fit:.3g}")
    plt.xlabel("time /s")
    plt.ylabel("conc /M")
    plt.show()